# Study 1 — Corpus Characterisation: Transition Structure & Sequence Analysis

Analyses the sequential structure, entropy, phase composition, and domain profile of 320 reasoning traces.

**Outputs:**
- `outputs/study1_analysis/tables/transition_top_bigrams.csv`
- `outputs/study1_analysis/tables/self_transition_rates.csv`
- `outputs/study1_analysis/tables/transition_entropy_stats.csv`
- `outputs/study1_analysis/tables/sequence_characteristics.csv`
- `outputs/study1_analysis/tables/phase_structure_summary.csv`
- `outputs/study1_analysis/tables/domain_comparison_bogdan.csv`
- `outputs/study1_analysis/figures/transition_matrix_micro.png`
- `outputs/study1_analysis/figures/transition_entropy_histogram.png`
- `outputs/study1_analysis/figures/transition_entropy_by_task.png`
- `outputs/study1_analysis/figures/transition_entropy_by_completion.png`
- `outputs/study1_analysis/figures/opening_categories.png`
- `outputs/study1_analysis/figures/closing_categories.png`
- `outputs/study1_analysis/figures/first_hypo_position.png`
- `outputs/study1_analysis/figures/phase_structure.png`
- `outputs/study1_analysis/figures/domain_comparison_bogdan.png`

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
from study1_helpers import *
import math
from collections import Counter
from scipy.stats import kruskal, mannwhitneyu
import scikit_posthocs as sp

traces = load_traces()
df = build_sentence_df(traces)
print(f'Corpus: {df.trace_key.nunique()} traces, {len(df):,} sentences')

Loaded 320 traces from C:\Users\drcha\Desktop\cot-faithfulness\outputs\traces_clean_coded


Corpus: 320 traces, 73,383 sentences


## §2a. Transition Probability Matrix

Build a 9×9 bigram transition matrix P(micro_label_{t+1} | micro_label_t), counting transitions within traces only.

In [2]:
section_header('2a. Transition Probability Matrix')

# Count bigram transitions within traces
bigram_counts = Counter()
for key, grp in df.groupby('trace_key'):
    labels = grp.sort_values('position_idx')['micro_label'].tolist()
    for a, b in zip(labels[:-1], labels[1:]):
        bigram_counts[(a, b)] += 1

# Build 9x9 count matrix
count_mat = pd.DataFrame(0, index=MICRO_LABELS, columns=MICRO_LABELS, dtype=int)
for (fr, to), cnt in bigram_counts.items():
    count_mat.loc[fr, to] = cnt

# Row-normalise to get probabilities
row_sums = count_mat.sum(axis=1)
trans_prob = count_mat.div(row_sums.where(row_sums > 0, other=1), axis=0)

# ── Heatmap ──
fig, ax = plt.subplots(figsize=(10, 8))
annot = trans_prob.apply(
    lambda col: col.map(lambda x: f'{x:.2f}' if x >= 0.05 else '')
)
sns.heatmap(
    trans_prob.values, ax=ax, cmap='YlOrRd', vmin=0, vmax=0.8,
    annot=annot.values, fmt='',
    xticklabels=MICRO_LABELS, yticklabels=MICRO_LABELS,
    linewidths=0.4,
)
ax.set_xlabel('Next label (t+1)', fontsize=11)
ax.set_ylabel('Current label (t)', fontsize=11)
ax.set_title('Micro-Label Transition Probabilities P(next | current)', fontsize=12)
plt.tight_layout()
save_fig(fig, 'transition_matrix_micro.png')
plt.show()

# ── Top 20 bigrams ──
top20 = sorted(bigram_counts.items(), key=lambda x: x[1], reverse=True)[:20]
top_rows = []
for (fr, to), cnt in top20:
    denom = row_sums.get(fr, 1)
    top_rows.append({
        'from_label': fr, 'to_label': to,
        'count': cnt,
        'probability': round(cnt / denom if denom > 0 else 0, 4),
    })
top_df = pd.DataFrame(top_rows)
save_table(top_df, 'transition_top_bigrams.csv')
display(top_df)

# ── Self-transition rates ──
self_rows = []
for lbl in MICRO_LABELS:
    self_rows.append({
        'label': lbl,
        'self_transition_rate': round(trans_prob.loc[lbl, lbl], 4),
        'self_transition_count': int(count_mat.loc[lbl, lbl]),
        'total_transitions_from': int(row_sums[lbl]),
    })
self_df = pd.DataFrame(self_rows)
save_table(self_df, 'self_transition_rates.csv')
display(self_df)


  2a. Transition Probability Matrix



  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\transition_matrix_micro.png
  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\tables\transition_top_bigrams.csv


,from_label,to_label,count,probability
0,TEST,TEST,25437,0.6921
1,HYPO,TEST,8032,0.6284
2,TEST,JUDGE,6316,0.1719
3,JUDGE,HYPO,5970,0.6622
4,DESCRIBE,DESCRIBE,4454,0.7262
5,TEST,HYPO,3340,0.0909
6,HYPO,HYPO,2021,0.1581
7,HYPO,JUDGE,1369,0.1071
8,PLAN,TEST,1279,0.4309
9,JUDGE,TEST,1126,0.1249


  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\tables\self_transition_rates.csv


,label,self_transition_rate,self_transition_count,total_transitions_from
0,ORIENT,0.5007,363,725
1,DESCRIBE,0.7262,4454,6133
2,SYNTHESIZE,0.2976,841,2826
3,HYPO,0.1581,2021,12781
4,TEST,0.6921,25437,36752
5,JUDGE,0.0924,833,9016
6,PLAN,0.1904,565,2968
7,MONITOR,0.1216,204,1677
8,RULE,0.5351,99,185


### Interpretation — Transition Matrix

The transition matrix reveals the dominant sequential patterns in the corpus. Key observations:

- **TEST→TEST** dominates with the highest self-transition rate, reflecting TEST's ~50% corpus share and the model's tendency to chain multiple evidence-checking steps.
- The **HYPO→TEST→JUDGE** cycle is visible as a strong sequential pattern: HYPO transitions frequently to TEST, TEST transitions to both TEST (self) and JUDGE, and JUDGE transitions back to HYPO (for another cycle) or to TEST.
- **DESCRIBE→DESCRIBE** shows moderate self-transition, reflecting the scanning phase where the model reads through panel features.
- **ORIENT** transitions primarily to DESCRIBE or PLAN, confirming its role as a trace-opening category.
- **RULE** is rarely a transition source (low frequency, typically trace-final).

## §2b. Transition Entropy

Shannon entropy of bigram and marginal distributions per trace. Higher entropy = more diverse reasoning patterns.

In [3]:
section_header('2b. Transition Entropy')

def shannon_entropy(counts_dict):
    """Compute Shannon entropy in bits from a counts dictionary."""
    total = sum(counts_dict.values())
    if total == 0:
        return 0.0
    ent = 0.0
    for c in counts_dict.values():
        if c > 0:
            p = c / total
            ent -= p * math.log2(p)
    return ent

# Compute per-trace entropies
entropy_rows = []
for key, grp in df.groupby('trace_key'):
    labels = grp.sort_values('position_idx')['micro_label'].tolist()
    meta = grp.iloc[0]
    
    # Transition entropy
    bigrams = Counter(zip(labels[:-1], labels[1:]))
    h_trans = shannon_entropy(bigrams)
    
    # Marginal entropy
    marginal = Counter(labels)
    h_marg = shannon_entropy(marginal)
    
    entropy_rows.append({
        'trace_key': key,
        'task_id': meta['task_id'],
        'set': meta['set'],
        'completed': meta['completed'],
        'n_sentences': len(labels),
        'transition_entropy': round(h_trans, 4),
        'marginal_entropy': round(h_marg, 4),
    })

ent_df = pd.DataFrame(entropy_rows)

# ── Statistical tests ──
# Kruskal-Wallis by task_id
groups_task = [grp['transition_entropy'].values for _, grp in ent_df.groupby('task_id')]
kw_stat, kw_p = kruskal(*groups_task)
print(f'Kruskal-Wallis (transition entropy ~ task): H={kw_stat:.3f}, p={kw_p:.4f}')

if kw_p < 0.05:
    dunn = sp.posthoc_dunn(ent_df, val_col='transition_entropy', group_col='task_id', p_adjust='bonferroni')
    print('\nDunn\'s post-hoc (Bonferroni-corrected p-values):')
    display(dunn.round(4))

# Mann-Whitney by completion status
completed_ent = ent_df[ent_df['completed'] == True]['transition_entropy']
truncated_ent = ent_df[ent_df['completed'] == False]['transition_entropy']
if len(completed_ent) > 0 and len(truncated_ent) > 0:
    mw_stat, mw_p = mannwhitneyu(completed_ent, truncated_ent, alternative='two-sided')
    print(f'\nMann-Whitney U (transition entropy ~ completion): U={mw_stat:.1f}, p={mw_p:.4f}')

# ── Descriptive stats table ──
def _desc(series, n):
    return {
        'mean': round(series.mean(), 4),
        'median': round(series.median(), 4),
        'sd': round(series.std(), 4),
        'min': round(series.min(), 4),
        'max': round(series.max(), 4),
        'n': n,
    }

stats_rows = []
for metric in ['transition_entropy', 'marginal_entropy']:
    stats_rows.append({'group': 'overall', 'metric': metric, **_desc(ent_df[metric], len(ent_df))})
    for tid in sorted(ent_df['task_id'].unique()):
        sub = ent_df[ent_df['task_id'] == tid][metric]
        stats_rows.append({'group': f'task{tid}', 'metric': metric, **_desc(sub, len(sub))})
    for status, label in [(True, 'completed'), (False, 'truncated')]:
        sub = ent_df[ent_df['completed'] == status][metric]
        if len(sub) > 0:
            stats_rows.append({'group': label, 'metric': metric, **_desc(sub, len(sub))})

stats_df = pd.DataFrame(stats_rows)
save_table(stats_df, 'transition_entropy_stats.csv')
display(stats_df)


  2b. Transition Entropy



Kruskal-Wallis (transition entropy ~ task): H=10.738, p=0.0132



Dunn's post-hoc (Bonferroni-corrected p-values):


,1,2,3,4
1,1.0000,0.1315,0.0113,1.0000
2,0.1315,1.0000,1.0000,1.0000
3,0.0113,1.0000,1.0000,0.4398
4,1.0000,1.0000,0.4398,1.0000



Mann-Whitney U (transition entropy ~ completion): U=16902.5, p=0.0000
  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\tables\transition_entropy_stats.csv


,group,metric,mean,median,sd,min,max,n
0,overall,transition_entropy,3.4561,3.4202,0.4282,2.4081,4.5835,320
1,task1,transition_entropy,3.5600,3.6175,0.4102,2.4339,4.3868,80
2,task2,transition_entropy,3.4122,3.3490,0.4692,2.4114,4.4109,80
3,task3,transition_entropy,3.3764,3.3004,0.3778,2.6395,4.3310,80
4,task4,transition_entropy,3.4758,3.4248,0.4353,2.4081,4.5835,80
5,completed,transition_entropy,3.5711,3.5697,0.4399,2.4081,4.5835,163
6,truncated,transition_entropy,3.3367,3.3095,0.3818,2.4114,4.3073,157
7,overall,marginal_entropy,2.1464,2.1378,0.2755,1.4776,2.9117,320
8,task1,marginal_entropy,2.2226,2.2560,0.2672,1.5146,2.8351,80
9,task2,marginal_entropy,2.1274,2.1136,0.2912,1.5216,2.7912,80


In [4]:
# ── Figures ──

# Histogram of transition entropy
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ent_df['transition_entropy'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('Transition Entropy (bits)', fontsize=11)
ax.set_ylabel('Number of Traces', fontsize=11)
ax.set_title('Distribution of Per-Trace Transition Entropy', fontsize=12)
plt.tight_layout()
save_fig(fig, 'transition_entropy_histogram.png')
plt.show()

# Boxplot by task
fig, ax = plt.subplots(figsize=(8, 5))
task_data = [ent_df[ent_df['task_id'] == t]['transition_entropy'] for t in sorted(ent_df['task_id'].unique())]
bp = ax.boxplot(task_data, labels=[f'Task {t}' for t in sorted(ent_df['task_id'].unique())])
ax.set_ylabel('Transition Entropy (bits)', fontsize=11)
ax.set_title('Transition Entropy by Task', fontsize=12)
ax.annotate(f'Kruskal-Wallis p={kw_p:.4f}', xy=(0.02, 0.95), xycoords='axes fraction', fontsize=10)
plt.tight_layout()
save_fig(fig, 'transition_entropy_by_task.png')
plt.show()

# Boxplot by completion
fig, ax = plt.subplots(figsize=(6, 5))
comp_data = [truncated_ent.values, completed_ent.values]
bp = ax.boxplot(comp_data, labels=['Truncated', 'Completed'])
ax.set_ylabel('Transition Entropy (bits)', fontsize=11)
ax.set_title('Transition Entropy by Completion Status', fontsize=12)
ax.annotate(f'Mann-Whitney p={mw_p:.4f}', xy=(0.02, 0.95), xycoords='axes fraction', fontsize=10)
plt.tight_layout()
save_fig(fig, 'transition_entropy_by_completion.png')
plt.show()

  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\transition_entropy_histogram.png


  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\transition_entropy_by_task.png
  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\transition_entropy_by_completion.png


C:\Users\drcha\AppData\Local\Temp\ipykernel_35620\3750225416.py:16: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(task_data, labels=[f'Task {t}' for t in sorted(ent_df['task_id'].unique())])
C:\Users\drcha\AppData\Local\Temp\ipykernel_35620\3750225416.py:27: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(comp_data, labels=['Truncated', 'Completed'])


### Interpretation — Transition Entropy

Transition entropy measures the diversity of sequential reasoning patterns within each trace. Higher entropy indicates more varied transitions between reasoning categories.

- In the MSc findings, low-buffer LLM conditions showed lower entropy (more stereotyped HYPO→TEST→JUDGE cycling). The entropy distribution here reveals whether DeepSeek-R1 shows similar stereotypy or more diverse reasoning.
- Differences by task may reflect task complexity: harder tasks (Task 3, conjunctive rules) may elicit more varied reasoning patterns.
- Completed vs truncated differences reveal whether traces that reach a conclusion show different sequential diversity from those that run out of tokens mid-exploration.

## §2c. Sequence Characteristics

Per-trace metrics: opening/closing labels, first HYPO position, distinct label count, and phase structure (scanning → cycling → convergence).

In [5]:
section_header('2c. Sequence Characteristics')

seq_rows = []
for key, grp in df.groupby('trace_key'):
    grp = grp.sort_values('position_idx')
    labels = grp['micro_label'].tolist()
    pos_norms = grp['position_norm'].tolist()
    meta = grp.iloc[0]
    n = len(labels)
    
    opening = labels[0]
    closing = labels[-1]
    
    # First HYPO position
    first_hypo_pos = np.nan
    first_hypo_idx = None
    for i, lbl in enumerate(labels):
        if lbl == 'HYPO':
            first_hypo_pos = pos_norms[i]
            first_hypo_idx = i
            break
    
    # Last JUDGE position AFTER first HYPO (not just last JUDGE overall)
    last_judge_idx = None
    if first_hypo_idx is not None:
        for i in range(n - 1, first_hypo_idx, -1):
            if labels[i] == 'JUDGE':
                last_judge_idx = i
                break
    
    n_distinct = len(set(labels))
    
    # Phase structure (three cases)
    if first_hypo_idx is None:
        # Case C: No HYPO — entire trace is scanning
        scanning_pct = 1.0
        cycling_pct = 0.0
        convergence_pct = 0.0
        strategy = 'direct_insight'
    elif last_judge_idx is not None:
        # Case A: Normal — HYPO exists and JUDGE exists after it
        scanning_pct = first_hypo_idx / n
        cycling_pct = (last_judge_idx - first_hypo_idx + 1) / n
        convergence_pct = (n - 1 - last_judge_idx) / n
        strategy = 'full_cycling'
    else:
        # Case B: HYPO but no JUDGE after it — cycling extends to end
        scanning_pct = first_hypo_idx / n
        cycling_pct = (n - 1 - first_hypo_idx + 1) / n
        convergence_pct = 0.0
        strategy = 'scan_test_conclude'
    
    seq_rows.append({
        'trace_key': key,
        'task_id': meta['task_id'],
        'set': meta['set'],
        'completed': meta['completed'],
        'opening_label': opening,
        'closing_label': closing,
        'first_hypo_pos': round(first_hypo_pos, 4) if not np.isnan(first_hypo_pos) else np.nan,
        'n_distinct_labels': n_distinct,
        'scanning_phase_pct': round(scanning_pct, 4),
        'cycling_phase_pct': round(cycling_pct, 4),
        'convergence_phase_pct': round(convergence_pct, 4),
        'reasoning_strategy': strategy,
    })

seq_df = pd.DataFrame(seq_rows)

# Verify phases sum to ~1.0 for all traces
seq_df['_phase_sum'] = seq_df['scanning_phase_pct'] + seq_df['cycling_phase_pct'] + seq_df['convergence_phase_pct']
assert (abs(seq_df['_phase_sum'] - 1.0) < 0.01).all(), \
    f"Phase sum check failed: {seq_df[abs(seq_df['_phase_sum'] - 1.0) >= 0.01][['trace_key', '_phase_sum']]}"
seq_df = seq_df.drop(columns=['_phase_sum'])

save_table(seq_df, 'sequence_characteristics.csv')
print(f'Sequence characteristics computed for {len(seq_df)} traces.')
display(seq_df[['first_hypo_pos', 'n_distinct_labels', 'scanning_phase_pct', 'cycling_phase_pct', 'convergence_phase_pct']].describe().round(3))

# Reasoning strategy summary
print('\n--- Reasoning Strategy ---')
print(seq_df['reasoning_strategy'].value_counts())
print(f"\nAll direct_insight traces completed: {seq_df[seq_df['reasoning_strategy']=='direct_insight']['completed'].all()}")
print(f"All scan_test_conclude traces completed: {seq_df[seq_df['reasoning_strategy']=='scan_test_conclude']['completed'].all()}")


  2c. Sequence Characteristics



  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\tables\sequence_characteristics.csv
Sequence characteristics computed for 320 traces.


,first_hypo_pos,n_distinct_labels,scanning_phase_pct,cycling_phase_pct,convergence_phase_pct
count,317.000,320.000,320.000,320.000,320.000
mean,0.177,8.306,0.183,0.792,0.025
std,0.205,0.717,0.215,0.223,0.045
min,0.014,5.000,0.014,0.000,0.000
25%,0.056,8.000,0.056,0.759,0.004
50%,0.099,8.000,0.099,0.872,0.011
75%,0.200,9.000,0.201,0.930,0.027
max,0.975,9.000,1.000,0.986,0.389



--- Reasoning Strategy ---
reasoning_strategy
full_cycling          303
scan_test_conclude     14
direct_insight          3
Name: count, dtype: int64

All direct_insight traces completed: True
All scan_test_conclude traces completed: True


In [6]:
# ── Opening label distribution ──
fig, ax = plt.subplots(figsize=(10, 5))
opening_counts = seq_df['opening_label'].value_counts()
opening_prop = opening_counts / len(seq_df)
cats_present = [c for c in MICRO_LABELS if c in opening_prop.index]
vals = [opening_prop.get(c, 0) for c in cats_present]
colours = [MICRO_COLOURS[c] for c in cats_present]
ax.bar(cats_present, vals, color=colours, edgecolor='white')
ax.set_ylabel('Proportion of Traces')
ax.set_xlabel('Opening Label')
ax.set_title('Distribution of Opening (First) Label Across Traces')
ax.set_xticklabels(cats_present, rotation=45, ha='right')
for i, v in enumerate(vals):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=9)
plt.tight_layout()
save_fig(fig, 'opening_categories.png')
plt.show()

# ── Closing label distribution ──
fig, ax = plt.subplots(figsize=(10, 5))
closing_counts = seq_df['closing_label'].value_counts()
closing_prop = closing_counts / len(seq_df)
cats_present = [c for c in MICRO_LABELS if c in closing_prop.index]
vals = [closing_prop.get(c, 0) for c in cats_present]
colours = [MICRO_COLOURS[c] for c in cats_present]
ax.bar(cats_present, vals, color=colours, edgecolor='white')
ax.set_ylabel('Proportion of Traces')
ax.set_xlabel('Closing Label')
ax.set_title('Distribution of Closing (Last) Label Across Traces')
ax.set_xticklabels(cats_present, rotation=45, ha='right')
for i, v in enumerate(vals):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=9)
plt.tight_layout()
save_fig(fig, 'closing_categories.png')
plt.show()

  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\opening_categories.png


C:\Users\drcha\AppData\Local\Temp\ipykernel_35620\1346735808.py:12: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(cats_present, rotation=45, ha='right')
C:\Users\drcha\AppData\Local\Temp\ipykernel_35620\1346735808.py:30: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(cats_present, rotation=45, ha='right')


  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\closing_categories.png


In [7]:
# ── First HYPO position histogram ──
fig, ax = plt.subplots(figsize=(10, 5))
hypo_pos = seq_df['first_hypo_pos'].dropna()
ax.hist(hypo_pos, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(hypo_pos.median(), color='red', linestyle='--', label=f'Median = {hypo_pos.median():.3f}')
ax.set_xlabel('Normalised Position (0=start, 1=end)')
ax.set_ylabel('Number of Traces')
ax.set_title('Position of First HYPO Sentence in Trace')
ax.legend()
plt.tight_layout()
save_fig(fig, 'first_hypo_position.png')
plt.show()
print(f'Traces with HYPO: {len(hypo_pos)}/{len(seq_df)}')
print(f'Mean first HYPO position: {hypo_pos.mean():.4f}')
print(f'Median first HYPO position: {hypo_pos.median():.4f}')

  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\first_hypo_position.png
Traces with HYPO: 317/320
Mean first HYPO position: 0.1774
Median first HYPO position: 0.0987


In [8]:
# ── Phase structure summary ──
phase_cols = ['scanning_phase_pct', 'cycling_phase_pct', 'convergence_phase_pct']

# Summary by task, completion, and reasoning strategy
summary_rows = []
for tid in sorted(seq_df['task_id'].unique()):
    sub = seq_df[seq_df['task_id'] == tid]
    row = {'group': f'Task {tid}', 'n': len(sub)}
    for col in phase_cols:
        row[f'{col}_mean'] = round(sub[col].mean(), 4)
        row[f'{col}_sd'] = round(sub[col].std(), 4)
    summary_rows.append(row)

for status, label in [(True, 'Completed'), (False, 'Truncated')]:
    sub = seq_df[seq_df['completed'] == status]
    if len(sub) > 0:
        row = {'group': label, 'n': len(sub)}
        for col in phase_cols:
            row[f'{col}_mean'] = round(sub[col].mean(), 4)
            row[f'{col}_sd'] = round(sub[col].std(), 4)
        summary_rows.append(row)

for strat in ['full_cycling', 'scan_test_conclude', 'direct_insight']:
    sub = seq_df[seq_df['reasoning_strategy'] == strat]
    if len(sub) > 0:
        row = {'group': strat, 'n': len(sub)}
        for col in phase_cols:
            row[f'{col}_mean'] = round(sub[col].mean(), 4)
            row[f'{col}_sd'] = round(sub[col].std(), 4)
        summary_rows.append(row)

phase_summary = pd.DataFrame(summary_rows)
save_table(phase_summary, 'phase_structure_summary.csv')
display(phase_summary)

# ── Stacked bar chart ──
fig, ax = plt.subplots(figsize=(10, 6))
groups = phase_summary['group'].tolist()
x = np.arange(len(groups))
scanning = phase_summary['scanning_phase_pct_mean'].values
cycling = phase_summary['cycling_phase_pct_mean'].values
convergence = phase_summary['convergence_phase_pct_mean'].values

ax.bar(x, scanning, label='Scanning (before 1st HYPO)', color='#4C72B0')
ax.bar(x, cycling, bottom=scanning, label='Cycling (1st HYPO -> last JUDGE)', color='#C44E52')
ax.bar(x, convergence, bottom=scanning + cycling, label='Convergence (after last JUDGE)', color='#CCB974')
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Proportion of Trace')
ax.set_title('Mean Phase Structure by Task, Completion, and Reasoning Strategy')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
save_fig(fig, 'phase_structure.png')
plt.show()

  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\tables\phase_structure_summary.csv


,group,n,scanning_phase_pct_mean,scanning_phase_pct_sd,cycling_phase_pct_mean,cycling_phase_pct_sd,convergence_phase_pct_mean,convergence_phase_pct_sd
0,Task 1,80,0.2117,0.2390,0.7689,0.2400,0.0194,0.0289
1,Task 2,80,0.1732,0.1884,0.8005,0.1994,0.0263,0.0397
2,Task 3,80,0.1053,0.0817,0.8672,0.1147,0.0276,0.0502
3,Task 4,80,0.2415,0.2770,0.7299,0.2835,0.0286,0.0557
4,Completed,163,0.2726,0.2653,0.6960,0.2725,0.0314,0.0568
5,Truncated,157,0.0898,0.0654,0.8909,0.0722,0.0193,0.0258
6,full_cycling,303,0.1509,0.1600,0.8222,0.1764,0.0269,0.0455
7,scan_test_conclude,14,0.7013,0.2672,0.2987,0.2672,0.0000,0.0000
8,direct_insight,3,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000


  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\phase_structure.png


### Interpretation — Sequence Characteristics

- **Opening moves:** Most traces open with ORIENT (restating the task) or DESCRIBE (beginning to scan the panels). This confirms the expected scanning phase at trace onset.
- **Closing moves:** Completed traces end with RULE; truncated traces end mid-reasoning (TEST, HYPO, or JUDGE).
- **First HYPO position:** The median position of the first hypothesis indicates how much scanning occurs before the model begins hypothesis-testing. Earlier first HYPO = shorter scanning phase.
- **Phase structure:** The scanning→cycling→convergence decomposition shows that the cycling phase (hypothesis testing) dominates most traces. Completed traces should show a convergence phase (post-JUDGE RULE articulation), while truncated traces have zero convergence.

## §2d. Domain Comparison to Bogdan et al.

Map our integrated taxonomy to Bogdan's 8 categories using the post-hoc mapping from Decision 24. Compare Zendo (visual inductive reasoning) proportions to Bogdan's math problem-solving baseline.

In [9]:
section_header('2d. Domain Comparison to Bogdan et al.')

# Bogdan baseline values (from their Figure 14, math problem-solving)
BOGDAN_BASELINE = {
    'problem_setup': 0.024,
    'plan_generation': 0.155,
    'fact_retrieval': 0.201,
    'active_computation': 0.327,
    'uncertainty_management': 0.140,
    'result_consolidation': 0.101,
    'self_checking': 0.045,
    'final_answer_emission': 0.007,
}

# Map our labels to Bogdan categories
# JUDGE needs to be split by judgement value
n_total = len(df)

# Count by micro_label
micro_counts = df['micro_label'].value_counts()

# Split JUDGE by judgement type
judge_df_sub = df[df['micro_label'] == 'JUDGE']
judge_accept = len(judge_df_sub[judge_df_sub['judgement'] == 'accept'])
judge_reject = len(judge_df_sub[judge_df_sub['judgement'] == 'reject'])
judge_uncertain = len(judge_df_sub[judge_df_sub['judgement'] == 'uncertain'])

# Build Zendo proportions mapped to Bogdan categories (pre-merge)
zendo_mapped = {
    'problem_setup': micro_counts.get('ORIENT', 0) / n_total,
    'plan_generation': (micro_counts.get('HYPO', 0) + micro_counts.get('PLAN', 0)) / n_total,
    'fact_retrieval': micro_counts.get('DESCRIBE', 0) / n_total,
    'active_computation': micro_counts.get('TEST', 0) / n_total,
    'uncertainty_management': (judge_reject + judge_uncertain + micro_counts.get('MONITOR', 0)) / n_total,
    'result_consolidation': (micro_counts.get('SYNTHESIZE', 0) + judge_accept) / n_total,
    'self_checking': 0.0,  # Absorbed into MONITOR
    'final_answer_emission': micro_counts.get('RULE', 0) / n_total,
}

mapping_notes = {
    'problem_setup': 'ORIENT',
    'plan_generation': 'HYPO + PLAN',
    'fact_retrieval': 'DESCRIBE',
    'active_computation': 'TEST',
    'uncertainty_management': 'JUDGE(reject+uncertain) + MONITOR',
    'result_consolidation': 'SYNTHESIZE + JUDGE(accept)',
    'self_checking': 'Absorbed into MONITOR (see note)',
    'final_answer_emission': 'RULE',
}

# Build initial comparison table
bogdan_cats = list(BOGDAN_BASELINE.keys())
comp_rows = []
for cat in bogdan_cats:
    comp_rows.append({
        'bogdan_category': cat,
        'zendo_proportion': round(zendo_mapped[cat], 4),
        'bogdan_proportion': BOGDAN_BASELINE[cat],
        'difference': round(zendo_mapped[cat] - BOGDAN_BASELINE[cat], 4),
        'mapping_source': mapping_notes[cat],
    })
comp_df = pd.DataFrame(comp_rows)

# Merge uncertainty_management and self_checking into a single row
um_row = comp_df[comp_df['bogdan_category'] == 'uncertainty_management'].iloc[0]
sc_row = comp_df[comp_df['bogdan_category'] == 'self_checking'].iloc[0]
merged_zendo = um_row['zendo_proportion'] + sc_row['zendo_proportion']
merged_bogdan = um_row['bogdan_proportion'] + sc_row['bogdan_proportion']
merged = {
    'bogdan_category': 'uncertainty_mgmt +\nself_checking',
    'zendo_proportion': round(merged_zendo, 4),
    'bogdan_proportion': round(merged_bogdan, 4),
    'difference': round(merged_zendo - merged_bogdan, 4),
    'mapping_source': 'JUDGE(reject/uncertain) + MONITOR',
}
comp_df = comp_df[~comp_df['bogdan_category'].isin(['uncertainty_management', 'self_checking'])]
comp_df = pd.concat([comp_df, pd.DataFrame([merged])], ignore_index=True)

save_table(comp_df, 'domain_comparison_bogdan.csv')
display(comp_df)

# Note about the merge
print(f'\nNote: Merged uncertainty_management ({um_row["bogdan_proportion"]}) + self_checking ({sc_row["bogdan_proportion"]}) = {merged_bogdan} on Bogdan side')
print(f'  Zendo side: JUDGE(reject/uncertain) + MONITOR = {merged_zendo:.4f}')
print(f'  Rationale: Our MONITOR absorbs both Bogdan functions (see Decision 24).')


  2d. Domain Comparison to Bogdan et al.

  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\tables\domain_comparison_bogdan.csv


,bogdan_category,zendo_proportion,bogdan_proportion,difference,mapping_source
0,problem_setup,0.0099,0.024,-0.0141,ORIENT
1,plan_generation,0.2153,0.155,0.0603,HYPO + PLAN
2,fact_retrieval,0.0836,0.201,-0.1174,DESCRIBE
3,active_computation,0.5019,0.327,0.1749,TEST
4,result_consolidation,0.0489,0.101,-0.0521,SYNTHESIZE + JUDGE(accept)
5,final_answer_emission,0.0045,0.007,-0.0025,RULE
6,uncertainty_mgmt +\nself_checking,0.1360,0.185,-0.0490,JUDGE(reject/uncertain) + MONITOR



Note: Merged uncertainty_management (0.14) + self_checking (0.045) = 0.185 on Bogdan side
  Zendo side: JUDGE(reject/uncertain) + MONITOR = 0.1360
  Rationale: Our MONITOR absorbs both Bogdan functions (see Decision 24).


In [10]:
# ── Grouped bar chart (merged categories) ──
fig, ax = plt.subplots(figsize=(12, 6))
plot_cats = comp_df['bogdan_category'].tolist()
x = np.arange(len(plot_cats))
w = 0.35
zendo_vals = comp_df['zendo_proportion'].values
bogdan_vals = comp_df['bogdan_proportion'].values

ax.bar(x - w/2, zendo_vals, w, label='Zendo (our corpus)', color='steelblue', alpha=0.85)
ax.bar(x + w/2, bogdan_vals, w, label='Bogdan et al. (math)', color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '\n') for c in plot_cats], rotation=0, ha='center', fontsize=8)
ax.set_ylabel('Proportion', fontsize=11)
ax.set_title('Domain Comparison: Zendo vs Bogdan et al. (Math)', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
save_fig(fig, 'domain_comparison_bogdan.png')
plt.show()

  Saved: C:\Users\drcha\Desktop\cot-faithfulness\outputs\study1_analysis\figures\domain_comparison_bogdan.png


### Interpretation — Domain Comparison

The Bogdan comparison reveals domain-driven differences in reasoning profiles:

- **active_computation** (TEST): Both domains show this as dominant, but Zendo's proportion reflects evidence-checking against visual panels rather than algebraic manipulation.
- **fact_retrieval** (DESCRIBE): Higher in math problem-solving, where fact retrieval involves recalling formulas and definitions. In Zendo, DESCRIBE captures panel-by-panel evidence extraction — reading off cone colours, sizes, orientations, and spatial relationships.
- **plan_generation** (HYPO + PLAN): The combination of HYPO and PLAN maps to Bogdan's plan_generation. In Zendo, hypothesis generation (proposing candidate rules) is a core activity distinct from the math domain where "plan" means selecting a solution strategy.
- **uncertainty_mgmt + self_checking**: Bogdan's `uncertainty_management` (14.0%) and `self_checking` (4.5%) are merged into a single combined category (18.5%) because our MONITOR category absorbs both functions — process uncertainty AND verification of prior steps. On the Zendo side, the combined value includes JUDGE(reject/uncertain) + MONITOR. This merged comparison is more accurate than showing self_checking at 0% for Zendo (see PHASE3 Decision 24).
- **final_answer_emission** (RULE): Very low in both domains, as expected — final answers are a small fraction of extended reasoning.

Some differences may also be model-driven (they used R1-Distill-Qwen-14B, we used R1-Distill-Llama-8B) rather than purely domain-driven.